In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [3]:
# Load pre-trained LLaMA model (small for demo)
# model_name = "decapoda-research/llama-7b-hf"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
#  Apply LoRA for efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)

In [5]:

# 3️⃣ Load dataset
dataset = load_dataset("json", data_files="iot_dataset.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
# Convert to LLM input-output format
def preprocess(example):
    text = f"Sensor readings: {example['input']} => Action: {example['output']}"
    tokenized = tokenizer(text, truncation=True, max_length=128)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(preprocess, batched=False)


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [7]:

# Training setup
training_args = TrainingArguments(
    output_dir="./llama_iot_demo",
    per_device_train_batch_size=1,  # adjust for GPU memory
    num_train_epochs=3,
    learning_rate=3e-4,
    logging_steps=5,
    fp16=True,
    save_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

# 6️⃣ Train
trainer.train()


  0%|          | 0/30 [00:00<?, ?it/s]

{'loss': 3.063, 'grad_norm': 3.183609962463379, 'learning_rate': 0.00025, 'epoch': 0.5}
{'loss': 2.4827, 'grad_norm': 3.448983669281006, 'learning_rate': 0.00019999999999999998, 'epoch': 1.0}
{'loss': 2.0813, 'grad_norm': 3.266525983810425, 'learning_rate': 0.00015, 'epoch': 1.5}
{'loss': 1.7069, 'grad_norm': 3.281580924987793, 'learning_rate': 9.999999999999999e-05, 'epoch': 2.0}
{'loss': 1.4288, 'grad_norm': 4.081394195556641, 'learning_rate': 4.9999999999999996e-05, 'epoch': 2.5}
{'loss': 1.3147, 'grad_norm': 3.0986809730529785, 'learning_rate': 0.0, 'epoch': 3.0}
{'train_runtime': 108.2691, 'train_samples_per_second': 0.277, 'train_steps_per_second': 0.277, 'train_loss': 2.0129026412963866, 'epoch': 3.0}


TrainOutput(global_step=30, training_loss=2.0129026412963866, metrics={'train_runtime': 108.2691, 'train_samples_per_second': 0.277, 'train_steps_per_second': 0.277, 'total_flos': 9898635497472.0, 'train_loss': 2.0129026412963866, 'epoch': 3.0})

In [11]:
prompt = """
Sensor readings:
Node 1: CPU 90%, RAM 80%, Temp 82C, Latency 220ms
=> Action:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    temperature=0.5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


/home/iqujeff/anaconda3/envs/openai/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



Sensor readings:
Node 1: CPU 90%, RAM 80%, Temp 82C, Latency 220ms
=> Action:
- Increase CPU utilization by 10%
- Reduce RAM usage by 10%
- Increase network latency by 20ms

Node 2: CPU 80%, RAM 90%, Temp 80C, Laten


In [12]:
prompt = """
Sensor readings:
Node 1: CPU 90%, RAM 80%, Temp 82C, Latency 220ms

Instructions: 
1. Provide a list of recommended actions.
2. Explain why.
3. Stop after Node 1 actions.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    temperature=0.5
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



Sensor readings:
Node 1: CPU 90%, RAM 80%, Temp 82C, Latency 220ms

Instructions: 
1. Provide a list of recommended actions.
2. Explain why.
3. Stop after Node 1 actions.
4. Repeat for remaining nodes.
5. Provide a summary of the overall performance.

Action 1: CPU 90%, RAM 80%, Temp 82C, Latency 220ms
- Node 1: Increase CPU
